# Pressure-Performance Ranking — T20 Cricket
**IPL 2022–2026 + T20Is + Minor T20 Tournaments**

Top-10 Batters and Top-10 Bowlers by pressure performance relative to their own normal baseline,
league-gap normalised and shrinkage-adjusted.

## Step 0 — Setup & Config

In [1]:
import gc, warnings, os
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

DATA = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data")
OUT  = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v2")
OUT.mkdir(exist_ok=True)

# Verify
assert DATA.exists(), f"Data dir not found: {DATA}"
print("Data files:", sorted(x.name for x in DATA.glob("*.csv")))

try:
    import psutil
    def mem(): return f"{round(psutil.Process().memory_info().rss/1e6)} MB"
except ImportError:
    def mem(): return "(psutil not installed)"
print("RAM:", mem())

Data files: ['ball_by_ball.csv', 'competition_team_player_mapping.csv', 'competitions.csv', 'matches.csv', 'playing_eleven.csv', 'teams.csv']
RAM: 119 MB


---
## Step 1 — Load & Clean

In [2]:
# ── ball_by_ball: only the columns we need ──
keep_cols = [
    "match_id", "innings", "overid", "innings_ball_num",
    "batting_team_id", "bowling_team_id",
    "striker_id", "striker_name", "bowler_id", "bowler_name",
    "runs", "is_wicket", "dismissal_type", "dismissed_player_id",
    "extras_type", "extras_2_runs",
    "pace_or_spin"
]
bbb = pd.read_csv(DATA / "ball_by_ball.csv", usecols=keep_cols)

# Keep only regular innings (drop Super Overs: innings 3, 4)
bbb = bbb[bbb["innings"].isin([1, 2])].copy()
print(f"Ball-by-ball: {bbb.shape}")

# ── Match context ──
matches = pd.read_csv(DATA / "matches.csv", usecols=[
    "match_id", "comp_id", "match_date", "stage", "knock_out",
    "revised_target", "first_inn_revised_overs", "second_inn_revised_overs",
    "venue_id", "win_team_id"
])
comps = pd.read_csv(DATA / "competitions.csv", usecols=[
    "comp_id", "match_format", "season"
])

# Merge
df = bbb.merge(matches, on="match_id", how="left")
df = df.merge(comps, on="comp_id", how="left")
del bbb; gc.collect()

df = df.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)

# ── Derived columns ──
# Legal ball = NOT wide, NOT Overstep (no-ball)
df["legal_ball"] = ~df["extras_type"].isin(["wide", "Overstep"])

# Extra penalty run for wide/no-ball (1 run each, plus extras_2_runs)
df["extra_run"] = df["extras_type"].isin(["wide", "Overstep"]).astype(int)

# Approximate team runs per ball (for reconstructing score)
# = bat runs + 1 for wide/noball + extras_2_runs (overthrows etc)
df["approx_runs"] = df["runs"].fillna(0).astype(int) + df["extra_run"] + df["extras_2_runs"].fillna(0).astype(int) + df["extras_type"].isin(["byes","legbyes"]).astype(int)

# Batter metrics (legal balls only later)
# runs_bat = runs (bat runs only; verified max=6)
df["runs_bat"] = df["runs"].fillna(0).astype(int)

# Striker dismissed on this ball?
df["striker_out"] = (df["is_wicket"].astype(bool)) & (df["dismissed_player_id"] == df["striker_id"])

# Bowler metrics
# bowler_runs = bat runs + wide/noball penalty + extras_2_runs, but byes/legbyes → 0
df["bowler_runs"] = df["runs"].fillna(0).astype(int) + df["extra_run"] + df["extras_2_runs"].fillna(0).astype(int)
df.loc[df["extras_type"].isin(["byes", "legbyes"]), "bowler_runs"] = 0

# bowler_wkt: exclude RunOut, Retire Out, Obstructing the field / Obstracting
non_bowler_dismissals = ["RunOut", "Retire Out", "Obstructing the field", "Obstracting"]
df["bowler_wkt"] = (df["is_wicket"].astype(bool)) & (~df["dismissal_type"].isin(non_bowler_dismissals))

# Format flag
df["is_t20i"] = (df["match_format"] == "T20I").astype(int)

print(f"Merged shape: {df.shape}, RAM: {mem()}")
print(f"Unique matches: {df['match_id'].nunique()}")
print(f"Format split: {df.groupby('match_format').size().to_dict()}")
print(f"runs max={df['runs'].max()}, min={df['runs'].min()}")
print("✓ Step 1 — Load & Clean complete")

Ball-by-ball: (200961, 17)
Merged shape: (200961, 36), RAM: 174 MB
Unique matches: 867
Format split: {'T20': 94646, 'T20I': 106315}
runs max=6, min=0
✓ Step 1 — Load & Clean complete


---
## Step 2 — Reconstruct Match State Per Ball

In [3]:
# ── Innings total balls (120 unless DLS-shortened) ──
df["inn_total_balls"] = 120
r1 = df["first_inn_revised_overs"]
r2 = df["second_inn_revised_overs"]
ok1 = (df["innings"] == 1) & r1.between(1, 20)
ok2 = (df["innings"] == 2) & r2.between(1, 20)
df.loc[ok1, "inn_total_balls"] = (r1[ok1] * 6).round().astype(int)
df.loc[ok2, "inn_total_balls"] = (r2[ok2] * 6).round().astype(int)

# ── Pre-ball state (cumulative BEFORE this ball) ──
g = df.groupby(["match_id", "innings"], sort=False)
df["score_before"] = g["approx_runs"].cumsum() - df["approx_runs"]
df["wkts_before"]  = g["is_wicket"].cumsum().astype(int) - df["is_wicket"].astype(int)
df["legal_before"] = g["legal_ball"].cumsum().astype(int) - df["legal_ball"].astype(int)
del g; gc.collect()

df["balls_left_before"] = (df["inn_total_balls"] - df["legal_before"]).clip(lower=0)

# ── Post-ball state ──
df["score_after"] = df["score_before"] + df["approx_runs"]
df["wkts_after"]  = df["wkts_before"]  + df["is_wicket"].astype(int)
df["legal_after"] = df["legal_before"] + df["legal_ball"].astype(int)
df["balls_left_after"] = (df["inn_total_balls"] - df["legal_after"]).clip(lower=0)

# ── Innings 2: target, runs needed, required rate ──
first_total = (df[df["innings"] == 1]
               .groupby("match_id")["approx_runs"].sum()
               .reset_index()
               .rename(columns={"approx_runs": "first_total"}))
df = df.merge(first_total, on="match_id", how="left")

df["target"] = df["first_total"] + 1
df.loc[df["revised_target"].notna(), "target"] = df.loc[df["revised_target"].notna(), "revised_target"]
df.loc[df["innings"] == 1, "target"] = np.nan

df["runs_needed_before"] = df["target"] - df["score_before"]
df["runs_needed_after"]  = df["target"] - df["score_after"]

# Required run rate (per over = *6/balls_left); guard div-by-zero
df["req_rate_before"] = np.where(
    (df["innings"] == 2) & (df["balls_left_before"] > 0),
    df["runs_needed_before"] * 6 / df["balls_left_before"], np.nan)
df["req_rate_after"] = np.where(
    (df["innings"] == 2) & (df["balls_left_after"] > 0),
    df["runs_needed_after"] * 6 / df["balls_left_after"], np.nan)

# Current run rate before
df["curr_rate_before"] = np.where(
    df["legal_before"] > 0,
    df["score_before"] * 6 / df["legal_before"], 0.0)

# ── bat_team_won ──
df["bat_team_won"] = (df["win_team_id"] == df["batting_team_id"]).astype(float)
df.loc[df["win_team_id"].isna(), "bat_team_won"] = np.nan

# ── Sanity checks ──
assert (df["balls_left_before"] >= 0).all(), "Negative balls_left_before found!"
assert (df["balls_left_before"] <= 120).all(), "balls_left_before > 120 found!"
assert (df["wkts_before"] >= 0).all(), "Negative wkts_before found!"
print(df[["score_before", "wkts_before", "balls_left_before"]].describe().round(1))
print(f"\nChase rows with missing target: {int(df.loc[df.innings==2, 'target'].isna().sum())}")

# ── Save intermediate state ──
df.to_pickle(OUT / "state.pkl")
print(f"\nSaved state.pkl ({round((OUT/'state.pkl').stat().st_size/1e6)} MB)")
print(f"RAM: {mem()}")
print("✓ Step 2 — Match State complete")

       score_before  wkts_before  balls_left_before
count      200961.0     200961.0           200961.0
mean           79.8          2.7               62.8
std            52.7          2.2               34.0
min             0.0          0.0                0.0
25%            36.0          1.0               34.0
50%            75.0          2.0               63.0
75%           118.0          4.0               92.0
max           301.0          9.0              120.0

Chase rows with missing target: 0

Saved state.pkl (76 MB)
RAM: 204 MB
✓ Step 2 — Match State complete


---
## Step 3 — Win-Probability Model (Leverage)

In [7]:
# ══════════════════════════════════════════════════════════════════════════
# Step 3 (v2) — Win-probability model, out-of-fold by match, EXPECTED leverage
# Replaces BOTH old cells 8 and 9.
# ══════════════════════════════════════════════════════════════════════════
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss

d0 = df[(df["balls_left_before"] > 0) & df["bat_team_won"].notna()].copy()
d0["phase"] = pd.cut(d0["overid"], [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)   # PP / middle / death
d0["pk"] = d0["is_t20i"].astype(int) * 3 + d0["phase"]                                # format x phase

# 1) What can happen on the next legal ball, by phase (empirical frequencies)
lg = d0[d0["legal_ball"]]
oc = np.where(lg["is_wicket"].astype(bool), "W", lg["runs_bat"].clip(upper=6).replace({5: 4}).astype(str))
P_OUT = lg.assign(o=oc).groupby("pk")["o"].value_counts(normalize=True).unstack().fillna(0)
OUTS = [("W", 0, 1)] + [(str(r), r, 0) for r in [0, 1, 2, 3, 4, 6]]      # (name, runs, wicket)

# 2) One model per innings (chosen on a time-based holdout: simple beats complex here)
CFG = {
    1: dict(feat=["score_before", "wkts_before", "balls_left_before", "overid", "is_t20i"],
            mk=lambda: HGB(max_depth=2, max_iter=120, learning_rate=0.05, random_state=42)),
    2: dict(feat=["runs_needed_before", "wkts_before", "balls_left_before", "is_t20i"],
            mk=lambda: HGB(max_iter=250, learning_rate=0.06, max_leaf_nodes=15, min_samples_leaf=200,
                           l2_regularization=1.0, monotonic_cst=[-1, -1, 0, 0], random_state=0)),
}
def shift_state(X, i, r, w, bl_drop):
    N = X.copy()
    if i == 1: N[:, 0] += r          # score up
    else:      N[:, 0] -= r          # runs needed down
    N[:, 1] += w                     # wickets
    N[:, 2] = np.clip(N[:, 2] - bl_drop, 0, None)   # balls left
    return N
def terminal(wp, N, i):              # chase won / lost is certain
    if i == 2:
        wp = np.where(N[:, 0] <= 0, 1.0, wp)
        wp = np.where((N[:, 1] >= 10) | ((N[:, 2] <= 0) & (N[:, 0] > 0)), 0.0, wp)
    return wp

SAVE = ["match_id", "innings", "innings_ball_num", "striker_id", "striker_name", "bowler_id", "bowler_name",
        "runs_bat", "approx_runs", "legal_ball", "is_wicket", "striker_out", "bowler_wkt", "bowler_runs",
        "batting_team_id", "bowling_team_id", "extras_type", "overid", "phase", "wkts_before", "score_before", "runs_needed_before",
        "balls_left_before", "pace_or_spin", "is_t20i", "wp_before", "wp_after", "leverage", "exp_dwp"]
for i in (1, 2):
    d = d0[d0["innings"] == i].copy()
    X = d[CFG[i]["feat"]].astype(float).values; y = d["bat_team_won"].astype(int).values
    wp_b, wp_a, lev, edw = (np.zeros(len(d)) for _ in range(4))
    for tr, te in GroupKFold(5).split(X, y, d["match_id"]):        # every match is test exactly once
        mdl = CFG[i]["mk"]().fit(X[tr], y[tr]); Xt = X[te]; T = d.iloc[te]
        p0 = mdl.predict_proba(Xt)[:, 1]
        Na = shift_state(Xt, i, T["approx_runs"].values, T["is_wicket"].astype(int).values, T["legal_ball"].astype(int).values)
        wp_a[te] = terminal(mdl.predict_proba(Na)[:, 1], Na, i)    # what actually happened
        L = np.zeros(len(te)); E = np.zeros(len(te))
        for name, r, w in OUTS:                                    # what COULD happen (pre-ball, outcome-free)
            Nh = shift_state(Xt, i, r, w, 1); p1 = terminal(mdl.predict_proba(Nh)[:, 1], Nh, i)
            pr = T["pk"].map(P_OUT[name]).values
            L += pr * np.abs(p1 - p0); E += pr * (p1 - p0)
        wp_b[te], lev[te], edw[te] = p0, L, E
    d["wp_before"], d["wp_after"], d["leverage"], d["exp_dwp"] = wp_b, wp_a, lev, edw
    print(f"Innings {i}: out-of-fold AUC {roc_auc_score(y, wp_b):.3f} | logloss {log_loss(y, wp_b):.3f} (naive 0.693)")
    cal = (pd.DataFrame({"pred": wp_b, "actual": y, "bin": pd.cut(wp_b, np.linspace(0, 1, 11), include_lowest=True)})
           .groupby("bin", observed=True).agg(pred=("pred", "mean"), actual=("actual", "mean"), n=("actual", "size")).round(3))
    print(cal.to_string()); print(d["leverage"].describe().round(4).to_string())
    d[SAVE].to_pickle(OUT / f"inn{i}_wp.pkl")
    del d; gc.collect()
print("✓ Step 3 (v2) complete")

Innings 1: out-of-fold AUC 0.681 | logloss 0.642 (naive 0.693)
                pred  actual      n
bin                                
(-0.001, 0.1]  0.088   0.086    139
(0.1, 0.2]     0.160   0.090   1894
(0.2, 0.3]     0.261   0.243   7960
(0.3, 0.4]     0.355   0.314  12645
(0.4, 0.5]     0.455   0.442  23879
(0.5, 0.6]     0.543   0.563  32785
(0.6, 0.7]     0.646   0.677  16709
(0.7, 0.8]     0.738   0.740   6779
(0.8, 0.9]     0.824   0.753    953
count    103743.0000
mean          0.0072
std           0.0062
min           0.0000
25%           0.0031
50%           0.0054
75%           0.0093
max           0.1161
Innings 2: out-of-fold AUC 0.890 | logloss 0.415 (naive 0.693)
                pred  actual      n
bin                                
(-0.001, 0.1]  0.031   0.042  21197
(0.1, 0.2]     0.149   0.175   8456
(0.2, 0.3]     0.249   0.239   6729
(0.3, 0.4]     0.348   0.345   6861
(0.4, 0.5]     0.448   0.451   6877
(0.5, 0.6]     0.551   0.559   6230
(0.6, 0.7]     0.650  

---
## Step 4 — Pressure Label

In [8]:
# Reload from pickles (crash-safe restart point)
inn1 = pd.read_pickle(OUT / "inn1_wp.pkl")
inn2 = pd.read_pickle(OUT / "inn2_wp.pkl")
wp = pd.concat([inn1, inn2], ignore_index=True)
del inn1, inn2; gc.collect()
print(f"Combined WP rows: {len(wp)}")

# ── Pressure = top 25% of leverage WITHIN each innings ──
wp["is_pressure"] = False
thresholds = {}
for inn in (1, 2):
    mask = wp["innings"] == inn
    q75 = wp.loc[mask, "leverage"].quantile(0.75)
    thresholds[inn] = q75
    wp.loc[mask, "is_pressure"] = wp.loc[mask, "leverage"] >= q75
    n_press = wp.loc[mask, "is_pressure"].sum()
    n_total = mask.sum()
    print(f"Innings {inn}: q75 threshold = {q75:.6f}, "
          f"pressure = {n_press} ({n_press/n_total*100:.1f}%), "
          f"normal = {n_total - n_press}")

print(f"\nOverall: {wp['is_pressure'].sum()} pressure balls, "
      f"{(~wp['is_pressure']).sum()} normal balls")

# ── Diagnostic: pressure vs normal separation ──
# Filter to legal balls only for rate calculations
legal = wp[wp["legal_ball"]].copy()

diag = legal.groupby("is_pressure").agg(
    balls=("runs_bat", "count"),
    mean_runs_per_ball=("runs_bat", "mean"),
    mean_wicket_rate=("is_wicket", "mean"),
    mean_bowler_runs=("bowler_runs", "mean"),
    mean_bowler_wkt_rate=("bowler_wkt", "mean"),
).round(4)
print("\n── Pressure vs Normal Separation (legal balls) ──")
print(diag)

p_rpb  = diag.loc[True,  "mean_runs_per_ball"]
n_rpb  = diag.loc[False, "mean_runs_per_ball"]
p_wr   = diag.loc[True,  "mean_wicket_rate"]
n_wr   = diag.loc[False, "mean_wicket_rate"]
print(f"\nRuns ratio (pressure/normal): {p_rpb/n_rpb:.2f}x")
print(f"Wicket ratio (pressure/normal): {p_wr/n_wr:.1f}x")
print("(Expected: ~2x runs, ~4-15x wicket rate)")

print("\n✓ Step 4 — Pressure Label complete")

Combined WP rows: 198844
Innings 1: q75 threshold = 0.009273, pressure = 25942 (25.0%), normal = 77801
Innings 2: q75 threshold = 0.025178, pressure = 23776 (25.0%), normal = 71325

Overall: 49718 pressure balls, 149126 normal balls

── Pressure vs Normal Separation (legal balls) ──
              balls  mean_runs_per_ball  mean_wicket_rate  mean_bowler_runs  \
is_pressure                                                                   
False        143132              1.3726            0.0540            1.3727   
True          47508              1.4599            0.0625            1.4601   

             mean_bowler_wkt_rate  
is_pressure                        
False                      0.0514  
True                       0.0584  

Runs ratio (pressure/normal): 1.06x
Wicket ratio (pressure/normal): 1.2x
(Expected: ~2x runs, ~4-15x wicket rate)

✓ Step 4 — Pressure Label complete


In [11]:
st["phase"] = pd.cut(st["overid"], [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)

In [13]:
import pandas as pd
from pathlib import Path
OUT = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v2")
st = pd.read_pickle(OUT / "state.pkl")
st["phase"] = pd.cut(st["overid"], [0, 6, 15, 20], labels=[0, 1, 2]).astype(int)
need = ["match_id","batting_team_id","striker_id","bowler_id","runs_bat",
        "legal_ball","phase","venue_id","comp_id","bat_team_won","is_t20i"]
print("missing:", [c for c in need if c not in st.columns])
print("phase counts:", st["phase"].value_counts().to_dict())

missing: []
phase counts: {1: 92499, 0: 64008, 2: 44454}


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell S — two-sided (Elo-style) skill model on ALL legal balls.
# Produces player_skills.csv with batting_skill and bowling_skill (runs/ball
# above average, ridge-shrunk, opponent- and role-adjusted).
# ══════════════════════════════════════════════════════════════════════════
import gc, numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold

OUT = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\Analysis_v2")
DATA = Path(r"C:\Users\Anmol Sriwastava\Desktop\RR_Analytics\data")

# ── 1. Start from the enriched frame (state.pkl has runs, extras, venue, season) ──
st = pd.read_pickle(OUT / "state.pkl")

# batting_order from playing_eleven (striker's slot)
pe = pd.read_csv(DATA / "playing_eleven.csv", usecols=["match_id","team_id","player_id","batting_order"])
pe = pe.rename(columns={"team_id":"batting_team_id","player_id":"striker_id","batting_order":"bat_order"})
st = st.merge(pe, on=["match_id","batting_team_id","striker_id"], how="left")

# canonical names from the player master
mp = (pd.read_csv(DATA / "competition_team_player_mapping.csv",
                  usecols=["player_id","player_name"])
        .drop_duplicates("player_id"))
name_of = mp.set_index("player_id")["player_name"].to_dict()

# phase already exists in state.pkl; season comes from competitions
if "season" not in st.columns:
    comps = pd.read_csv(DATA / "competitions.csv", usecols=["comp_id","season"])
    st = st.merge(comps, on="comp_id", how="left")

# ── 2. Fit frame: legal balls only, one row per ball ──
F = st[st["legal_ball"]].copy()
F["y"] = F["runs_bat"].astype(float)                       # runs off the bat
F["bat_order"] = F["bat_order"].fillna(6).astype(int).clip(1, 11)
F["season_c"]  = F["season"].astype(str)
F = F[F["bat_team_won"].notna()].copy()                    # keep only result-known matches

# ── 3. One-hot design: striker, bowler, and the context controls ──
def design(frame, strikers, bowlers, orders, seasons, venues):
    n = len(frame)
    X = np.zeros((n, 2 + 1 + 3 + len(seasons) + len(venues)), dtype=np.float32)
    X[:, 0] = frame["striker_id"].map(strikers).fillna(-1).values
    X[:, 1] = frame["bowler_id"].map(bowlers).fillna(-1).values
    # context block
    ctx = pd.get_dummies(frame["bat_order"].astype(int), prefix="bo").reindex(
        columns=[f"bo_{i}" for i in range(1, 12)], fill_value=0).values
    ph  = pd.get_dummies(frame["phase"].astype(int), prefix="ph").reindex(
        columns=["ph_0","ph_1","ph_2"], fill_value=0).values
    sea = pd.get_dummies(frame["season_c"], prefix="se").reindex(
        columns=[f"se_{s}" for s in seasons], fill_value=0).values
    ven = pd.get_dummies(frame["venue_id"].astype(int), prefix="ve").reindex(
        columns=[f"ve_{v}" for v in venues], fill_value=0).values
    X[:, 2:2+11]  = ctx
    X[:, 2+11:2+14] = ph
    X[:, 2+14:2+14+len(seasons)] = sea
    X[:, 2+14+len(seasons):] = ven
    # fill striker/bowler -1 with zeros (unknown player) — ridge handles
    X[:, 0] = np.where(X[:, 0] < 0, 0, X[:, 0] + 1)        # +1 offset so 0 = baseline
    X[:, 1] = np.where(X[:, 1] < 0, 0, X[:, 1] + 1)
    return X

striker_ids = sorted(F["striker_id"].unique())
bowler_ids  = sorted(F["bowler_id"].unique())
seasons     = sorted(F["season_c"].unique())
venues      = sorted(F["venue_id"].dropna().astype(int).unique())
sid = {v: i for i, v in enumerate(striker_ids)}
bid = {v: i for i, v in enumerate(bowler_ids)}

# ── 4. Ridge with GroupKFold, alpha chosen on held-out matches ──
# GroupKFold for the sanity check only; final fit on all balls with the chosen alpha.
def fit_ridge(alpha, X, y, groups):
    """Fit on all balls except one held-out match fold; return out-of-fold MSE."""
    oof = np.full(len(y), np.nan)
    for tr, te in GroupKFold(4).split(X, y, groups):
        m = Ridge(alpha=alpha, fit_intercept=True, solver="lsqr")
        m.fit(X[tr], y[tr]); oof[te] = m.predict(X[te])
    return np.nanmean((oof - y) ** 2)

X = design(F, sid, bid, None, seasons, venues)
y = F["y"].values
groups = F["match_id"].values
print(f"skill frame: {len(F):,} legal balls, {len(striker_ids)} strikers, {len(bowler_ids)} bowlers")

alphas = [1.0, 5.0, 20.0, 50.0]
mse = {a: fit_ridge(a, X, y, groups) for a in alphas}
best_alpha = min(mse, key=mse.get)
print("alpha grid:", {k: round(v, 4) for k, v in mse.items()}, "-> chosen", best_alpha)

# ── 5. Final fit on all balls ──
mdl = Ridge(alpha=best_alpha, fit_intercept=True, solver="lsqr")
mdl.fit(X, y)
coef = mdl.coef_
bat_skill_raw = pd.Series(coef[1:1+len(striker_ids)], index=striker_ids)
bowl_skill_raw = pd.Series(coef[1+len(striker_ids):1+len(striker_ids)+len(bowler_ids)], index=bowler_ids)

# centre on the league mean (baseline intercept already absorbs it, but centre again for interpretability)
bat_skill = (bat_skill_raw - bat_skill_raw.mean()).rename("batting_skill")
bowl_skill = (bowl_skill_raw - bowl_skill_raw.mean()).rename("bowling_skill")

skills = pd.concat([bat_skill, bowl_skill], axis=1).reset_index().rename(columns={"index": "player_id"})
skills["player_name"] = skills["player_id"].map(name_of)
skills = skills.rename(columns={"batting_skill": "bat_skill_per_ball",
                                "bowling_skill": "bowl_skill_per_ball"})

# exposure counts for the report
nbat = F.groupby("striker_id").size().rename("n_balls_bat")
nbow = F.groupby("bowler_id").size().rename("n_balls_bowl")
skills = skills.merge(nbat.rename_axis("player_id").reset_index(), on="player_id", how="left")
skills = skills.merge(nbow.rename_axis("player_id").reset_index(), on="player_id", how="left")
skills = skills.fillna({"n_balls_bat": 0, "n_balls_bowl": 0})

skills.to_csv(OUT / "player_skills.csv", index=False)
print(f"saved {OUT/'player_skills.csv'}")

# ── 6. Sanity check: top-20 batting and bowling skills (n>=100 balls) ──
print("\n── TOP 20 BATTING SKILL (runs/ball above average, n≥100) ──")
topb = (skills[skills["n_balls_bat"] >= 100]
        .sort_values("bat_skill_per_ball", ascending=False)
        .head(20)[["player_name","bat_skill_per_ball","n_balls_bat"]].round(4))
print(topb.to_string(index=False))

print("\n── TOP 20 BOWLING SKILL (runs/ball conceded below average, n≥100) ──")
topw = (skills[skills["n_balls_bowl"] >= 100]
        .sort_values("bowl_skill_per_ball")            # lower = better
        .head(20)[["player_name","bowl_skill_per_ball","n_balls_bowl"]].round(4))
print(topw.to_string(index=False))

# ── 7. Within-format sanity: split by is_t20i and correlate ──
t20i = F.groupby("striker_id")["is_t20i"].mean()
skills["share_t20i"] = skills["player_id"].map(t20i)
print("\nplayers with both formats (share_t20i between 0.2 and 0.8):",
      int(((skills.share_t20i > 0.2) & (skills.share_t20i < 0.8)).sum()))
gc.collect()
print("✓ Skill cell complete — STOP here, inspect top-20s before propagating to Step 3")

---
## Step 5 — Player Ratings

In [9]:
# ══════════════════════════════════════════════════════════════════════════
# CELL A — Step 5 (v2): per-ball value = win probability ADDED ABOVE EXPECTED,
#          then empirical-Bayes shrinkage. Replaces old cells 13, 14, 15.
# ══════════════════════════════════════════════════════════════════════════
MIN_P = 100                                   # min pressure balls to be ranked
wv = wp.copy()
# ── Comparable-strength matches only: IPL, or T20I between two ICC full members (no associates / A-team / emerging events) ──
FULL = {"India", "Australia", "England", "South Africa", "New Zealand", "Pakistan", "Sri Lanka", "Bangladesh", "West Indies", "Afghanistan", "Zimbabwe", "Ireland"}
_mm = (pd.read_csv(DATA / "matches.csv", usecols=["match_id", "comp_id"])
       .merge(pd.read_csv(DATA / "competitions.csv", usecols=["comp_id", "comp_name"]), on="comp_id"))
_tn = pd.read_csv(DATA / "teams.csv").drop_duplicates("team_id").set_index("team_id")["team_name"]
_t = wv.drop_duplicates("match_id")[["match_id", "batting_team_id", "bowling_team_id"]].merge(_mm, on="match_id")
_t["ipl"] = _t.comp_name.str.startswith("IPL"); _t["minor"] = _t.comp_name.str.contains("Emerging|Rising")
_t["both_full"] = _t.batting_team_id.map(_tn).isin(FULL) & _t.bowling_team_id.map(_tn).isin(FULL)
_keep = set(_t.loc[_t.ipl | (~_t.minor & _t.both_full), "match_id"])
print(f"matches kept {len(_keep)} of {_t.match_id.nunique()} | balls kept {wv.match_id.isin(_keep).mean():.1%}")
wv = wv[wv.match_id.isin(_keep)].copy()
et_ = wv["extras_type"].fillna("none")
wv["dwp"]    = (wv["wp_after"] - wv["wp_before"]) * 100     # batting side's change in win-prob (WP points)
wv["excess"] = wv["dwp"] - wv["exp_dwp"] * 100              # vs what an average player yields from this exact state
neutral = et_.isin(["byes", "legbyes"])                     # nobody's skill -> excluded
wk_ = wv["is_wicket"].astype(bool)

# Batter: faces legal balls; blamed/credited only for his own dismissal
bat0 = wv[wv["legal_ball"] & ~neutral].copy()
bat0["x"] = bat0["excess"]
bat0.loc[bat0["is_wicket"].astype(bool) & ~bat0["striker_out"].astype(bool), "x"] = 0.0   # non-striker run-out
bat0["pid"] = bat0["striker_id"]
# Bowler: all deliveries (wides / no-balls count against him); zero-sum with the batter
bow0 = wv[~neutral].copy()
bow0["x"] = -bow0["excess"]
bow0.loc[bow0["is_wicket"].astype(bool) & ~bow0["bowler_wkt"].astype(bool), "x"] = 0.0     # run-out: not his wicket
bow0["pid"] = bow0["bowler_id"]
print("mean excess (should be ~0):  bat %.4f | bowl %.4f" % (bat0.x.mean(), bow0.x.mean()))

def label(q):  # pressure = top (1-q) of pre-ball EXPECTED leverage within each innings
    return wv.groupby("innings")["leverage"].transform(lambda s: s >= s.quantile(q))

def prep(frame, lab):
    f = frame.assign(is_p=lab.loc[frame.index].values)
    p, n = f[f.is_p], f[~f.is_p]
    pm = p.groupby(["match_id", "pid"])["x"].agg(n="size", sx="sum").reset_index()
    nm = n.groupby(["match_id", "pid"])["x"].agg(nn="size", snx="sum").reset_index()
    return pm, nm, p["x"].var()

def agg_players(pm, nm, wt=None):
    if wt is not None:                                         # bootstrap multiplicities
        fp = pm["match_id"].map(wt).fillna(0).values; fn = nm["match_id"].map(wt).fillna(0).values
        pm = pm.assign(n=pm.n * fp, sx=pm.sx * fp); nm = nm.assign(nn=nm.nn * fn, snx=nm.snx * fn)
    t = pm.groupby("pid")[["n", "sx"]].sum().join(nm.groupby("pid")[["nn", "snx"]].sum(), how="left").fillna(0)
    t = t[t.n > 0].copy(); t["P"] = t.sx / t.n
    t["N"] = np.where(t.nn > 0, t.snx / t.nn.replace(0, np.nan), np.nan)
    return t

def eb_fit(t, sig2, model, minn=15):
    """true pressure value theta_i ~ N(prior_i, tau^2); observed P_i ~ N(theta_i, sig2/n_i).
       M0: prior = league mean.  M1: prior = a + b*N_i (player's own normal-ball level)."""
    t = t.copy(); t["s2"] = sig2 / t.n; e = t[(t.n >= minn) & t.N.notna()]
    tau2 = max(e.P.var() - e.s2.mean(), 1e-9)
    for _ in range(5):
        wt = 1 / (tau2 + e.s2)
        if model == "M1":
            sw = np.sqrt(wt.values); a, bb = np.linalg.lstsq(np.c_[np.ones(len(e)), e.N] * sw[:, None], e.P.values * sw, rcond=None)[0]
        else:
            a, bb = np.average(e.P, weights=wt), 0.0
        tau2 = max(np.average((e.P - (a + bb * e.N)) ** 2 - e.s2, weights=wt), 1e-9)
    t["prior"] = a + bb * t.N.fillna(t.N.mean()) if model == "M1" else a
    t["B"] = tau2 / (tau2 + t.s2); t["post"] = t.prior + t.B * (t.P - t.prior); t["psd"] = np.sqrt(t.B * t.s2)
    return t

def pick_model(pm, nm, sig2, n_rep=30, seed=0):
    rng = np.random.default_rng(seed); ms = pm.match_id.unique(); err = {"raw": [], "M0": [], "M1": []}
    for _ in range(n_rep):                                     # predict one random half of matches from the other
        h = set(rng.choice(ms, len(ms) // 2, replace=False))
        A = agg_players(pm[pm.match_id.isin(h)], nm[nm.match_id.isin(h)]); B = agg_players(pm[~pm.match_id.isin(h)], nm[~nm.match_id.isin(h)])
        for m in ("M0", "M1"):
            j = eb_fit(A, sig2, m).join(B[["P", "n"]], rsuffix="B", how="inner"); j = j[(j.n >= 15) & (j.nB >= 15)]
            err[m].append(np.average((j.post - j.PB) ** 2, weights=j.nB))
            if m == "M0": err["raw"].append(np.average((j.P - j.PB) ** 2, weights=j.nB))
    sc = {k: float(np.mean(v)) for k, v in err.items()}
    return min(("M0", "M1"), key=sc.get), sc

def run(frame, namecol, q=0.75, model=None):
    pm, nm, sig2 = prep(frame, label(q))
    if model is None: model, sc = pick_model(pm, nm, sig2)
    else: sc = None
    T = eb_fit(agg_players(pm, nm), sig2, model)
    T = T[T.n >= MIN_P].copy()
    T["name"] = frame.groupby("pid")[namecol].agg(lambda s: s.mode()[0]).reindex(T.index)
    T["pressure_matches"] = pm.groupby("pid")["match_id"].nunique().reindex(T.index)
    T["lo90"] = T.post - 1.645 * T.psd; T["hi90"] = T.post + 1.645 * T.psd
    return dict(T=T.sort_values("post", ascending=False), pm=pm, nm=nm, sig2=sig2, model=model, score=sc)

RES = {"bat": run(bat0, "striker_name"), "bowl": run(bow0, "bowler_name")}
for k, R in RES.items():
    print(f"\n[{k}] model {R['model']} | held-out MSE  raw {R['score']['raw']:.3f}  M0 {R['score']['M0']:.3f}  M1 {R['score']['M1']:.3f} | eligible {len(R['T'])}")
    print(R["T"][["name", "n", "pressure_matches", "P", "N", "post", "lo90", "hi90"]].head(12).round(3).to_string())
bt_elig, bo_elig = RES["bat"]["T"], RES["bowl"]["T"]

# ══════════════════════════════════════════════════════════════════════════
# CELL B — Bootstrap over matches + sensitivity (replaces old cells 16-22)
# ══════════════════════════════════════════════════════════════════════════
N_BOOT = 300
RNG = np.random.default_rng(7)
for k, R in RES.items():
    ms = R["pm"].match_id.unique(); hits = pd.Series(0, index=R["T"].index)
    for _ in range(N_BOOT):
        wt = pd.Series(RNG.choice(ms, len(ms))).value_counts().to_dict()
        Tb = eb_fit(agg_players(R["pm"], R["nm"], wt), R["sig2"], R["model"])
        hits.loc[hits.index.intersection(Tb[Tb.n >= MIN_P].post.nlargest(10).index)] += 1
    R["T"]["boot_top10_pct"] = (hits / N_BOOT * 100).round(0)
    base = set(R["T"].head(10).index); frame = bat0 if k == "bat" else bow0; nc = "striker_name" if k == "bat" else "bowler_name"
    print(f"\n[{k}] sensitivity: overlap of top-10 with the base list")
    for q in (0.70, 0.80):                                         # pressure = top 30% / top 20%
        t10 = set(run(frame, nc, q, R["model"])["T"].head(10).index); print(f"   pressure top {round((1-q)*100)}%: {len(t10 & base)}/10")
    MIN_SAVE = MIN_P
    for mp in (60, 150):
        MIN_P = mp; t10 = set(run(frame, nc, 0.75, R["model"])["T"].head(10).index); MIN_P = MIN_SAVE
        print(f"   min pressure balls {mp}: {len(t10 & base)}/10")
bt_elig, bo_elig = RES["bat"]["T"], RES["bowl"]["T"]

# ══════════════════════════════════════════════════════════════════════════
# CELL C — Outputs (replaces old cells 24, 25, 26)
# ══════════════════════════════════════════════════════════════════════════
cols = ["name", "n", "pressure_matches", "P", "N", "post", "lo90", "hi90", "boot_top10_pct"]
for k, fn in (("bat", "top10_batters"), ("bowl", "top10_bowlers")):
    top = RES[k]["T"][cols].head(10).round(3).copy(); top.index.name = "player_id"; top.insert(0, "rank", range(1, 11))
    top.to_csv(OUT / f"{fn}.csv"); RES[k]["T"].to_csv(OUT / f"{k}_all_eligible.csv")
    print("═" * 70, f"\nTOP 10 {'BATTERS' if k == 'bat' else 'BOWLERS'} UNDER PRESSURE\n", "═" * 70); print(top.to_string())
lg_ = wp[wp["legal_ball"]]
print("\nPressure thresholds (expected leverage q75):", {i: round(float(wp.loc[wp.innings == i, 'leverage'].quantile(.75)), 4) for i in (1, 2)})
print("Eligible: batters", len(bt_elig), "| bowlers", len(bo_elig), "(>= %d pressure balls)" % MIN_P)

matches kept 761 of 842 | balls kept 90.6%
mean excess (should be ~0):  bat 0.0455 | bowl -0.0836

[bat] model M0 | held-out MSE  raw 0.650  M0 0.333  M1 0.334 | eligible 132
                       name    n  pressure_matches      P      N   post   lo90   hi90
pid                                                                                  
4708            Rinku Singh  399                55  1.019  0.116  0.754  0.396  1.113
6396              Tim David  416                67  0.837  0.135  0.629  0.275  0.982
2756         Shashank Singh  121                20  1.323  0.020  0.619  0.108  1.129
3926          Rahul Tewatia  207                35  1.010 -0.162  0.607  0.162  1.052
143          Dinesh Karthik  258                41  0.872  0.154  0.570  0.154  0.986
8101        Ashutosh Sharma  123                16  1.104 -0.050  0.525  0.017  1.034
5939          Cameron Green  293                46  0.749  0.013  0.512  0.113  0.912
3647           Shreyas Iyer  426                62 